# Customer Data Quality & Acquisition Analysis

## Project overview

This project analyses customer sign-up data to assess **data quality**, understand **customer acquisition patterns**, and generate **business-focused recommendations**.

The workflow is:

**Business questions → data quality assessment → cleaning & validation → exploratory analysis → business questions → findings → recommendations**

### Business questions

1. Which acquisition source generated the most customer signups in the latest month with recorded valid activity?
2. Which region has the highest proportion of incomplete customer records?
3. How does marketing opt-in behaviour vary across age groups?
4. Which subscription plan is most common overall, and how do plan preferences vary across age groups?

### Objectives

- Assess completeness and consistency of customer data.
- Identify duplicates, invalid dates, malformed emails and implausible ages.
- Apply transparent, documented cleaning rules.
- Analyse acquisition, demographics, marketing opt-in behaviour and plan selection.
- Translate analytical results into practical recommendations.

### Tools

- Python
- Pandas
- Matplotlib
- Google Colab

> **Portfolio/privacy note:** Only publish a synthetic or fully anonymised dataset. Do not upload real customer names, email addresses or other personally identifiable information.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Charts generated by this notebook are saved here for the GitHub portfolio.
OUTPUT_DIR = Path("visualisations")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Pandas version: {pd.__version__}")

## 1. Load the data

The notebook supports the original Google Colab upload workflow. For the GitHub version, the recommended repository structure is:

```text
customer-data-quality-acquisition-analysis/
├── data/
│   └── customer_signups.csv
├── notebooks/
│   └── customer_data_analysis.ipynb
└── visualisations/
```

If the approved dataset is not present locally, Colab will prompt for an upload.

In [ ]:
# Prefer the repository data folder, then the notebook working directory.
DATA_PATH = Path("data/customer_signups.csv")
ROOT_DATA_PATH = Path("customer_signups.csv")

if DATA_PATH.exists():
    data_file = DATA_PATH
elif ROOT_DATA_PATH.exists():
    data_file = ROOT_DATA_PATH
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        data_file = Path(uploaded_name)
    except ImportError as exc:
        raise FileNotFoundError(
            "customer_signups.csv was not found. Place it in data/customer_signups.csv "
            "or run this notebook in Colab and upload the approved dataset."
        ) from exc

df = pd.read_csv(data_file)
df_raw = df.copy()

required_columns = {
    "customer_id", "name", "email", "signup_date", "age",
    "gender", "region", "source", "plan_selected", "marketing_opt_in"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print(f"Loaded: {data_file}")
print(f"Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
df.columns  # list the columns in the dataset


In [ ]:
df.dtypes #show the types of column

In [ ]:
df.info()

In [ ]:
df.head() # list the top 5 rows of the dataset

In [ ]:
df.describe() # Summary statistics for numeric columns

## 2. Initial data quality assessment

### Missing values

In [ ]:
df.isnull().sum()

### Signup-date validation

In [ ]:
# Inspect values that cannot be parsed as dates before conversion.
parsed_dates = pd.to_datetime(df["signup_date"], format="mixed", errors="coerce")
invalid_date_rows = df.loc[
    parsed_dates.isna() & df["signup_date"].notna(),
    ["customer_id", "signup_date"]
]
display(invalid_date_rows)

# Convert invalid/unparseable values to NaT.
df["signup_date"] = parsed_dates
invalid_date_count = int(df["signup_date"].isna().sum())
print(f"Invalid/unparseable signup dates: {invalid_date_count}")

In [ ]:
# The date conversion is performed in the validation cell above.

In [ ]:
df.head(10)

In [ ]:
df['signup_date'].isna().sum()

In [ ]:
df[df['signup_date'].isna()]

In [ ]:
#Now all missing value across all dataset
df.isna().sum()

In [ ]:
# Future dates are inspected and retained for data-quality review.
future_dates = df.loc[
    df["signup_date"].notna() & (df["signup_date"] > analysis_date),
    ["customer_id", "signup_date"]
].sort_values("signup_date")
display(future_dates)

In [ ]:
analysis_date = pd.Timestamp.today().normalize()

future_date_mask = df["signup_date"] > analysis_date
future_dates = df.loc[
    future_date_mask,
    ["customer_id", "signup_date"]
].sort_values("signup_date")

display(future_dates)
print(f"Future-dated records flagged: {future_date_mask.sum()}")

## 3. Data cleaning and standardisation

In [ ]:
#Examining the entries for the Plant_selected variables
df['plan_selected'].value_counts(dropna=False)

In [ ]:
df['plan_selected'].unique()

In [ ]:
df['plan_selected'] = df['plan_selected'].str.strip() #removes white space from the entries

In [ ]:
df['plan_selected'].unique()

In [ ]:
df['plan_selected'] = df['plan_selected'].str.title() # Standardised capitalization

In [ ]:
df['plan_selected'].unique()

In [ ]:
# Correct the spelling error
df['plan_selected'] = df['plan_selected'].replace({
    'Premuim': 'Premium'
})
df['plan_selected'].unique()

### Gender standardisation

In [ ]:
df['gender'].value_counts(dropna=False)

In [ ]:
for value in df['gender'].unique():
    print(repr(value))

In [ ]:
df['gender'] = df['gender'].str.strip()
df['gender'] = df['gender'].str.title()
df['gender'].unique()

### Categorical and field audits

In [ ]:
df['source'].value_counts(dropna=False)

In [ ]:
df['region'].value_counts(dropna=False)

In [ ]:
df['marketing_opt_in'].value_counts(dropna=False)

In [ ]:
df['name'].isna().sum()

In [ ]:
for value in df['name'].dropna().unique(): #check for leading and trailing space
    if value != value.strip():
        print(repr(value))

In [ ]:
for value in df['email'].dropna().unique():#check for leading and trailing space
    if value != value.strip():
        print(repr(value))

In [ ]:
import re #Checking for malformed email address

email_pattern = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'

invalid_emails = df[
    df['email'].notna() &
    ~df['email'].str.match(email_pattern, na=False)
]

invalid_emails[['customer_id', 'email']]

### Duplicate customer records

In [ ]:
df['customer_id'].duplicated().sum()

In [ ]:
duplicate_ids = df[df['customer_id'].duplicated(keep=False)]

duplicate_ids['customer_id'].value_counts()

In [ ]:
duplicate_ids.sort_values('customer_id')# Sort to place the duplicate cstomer ids side by side to confirm that they are duplicated records

In [ ]:
records_before_dedup = len(df)

# Retain the most recent signup record for each customer.
df = (
    df.sort_values("signup_date", ascending=False, na_position="last")
      .drop_duplicates("customer_id", keep="first")
      .copy()
)

duplicate_rows_removed = records_before_dedup - len(df)
print(f"Records before deduplication: {records_before_dedup:,}")
print(f"Duplicate rows removed: {duplicate_rows_removed:,}")
print(f"Records after deduplication: {len(df):,}")
print(f"Duplicate customer IDs remaining: {df['customer_id'].duplicated().sum()}")

In [ ]:
df['customer_id'].duplicated().sum() #checking to see if there is any duplicate.

### Missing-value handling

In [ ]:
df.isna().sum()

In [ ]:
(df.isna().sum() / len(df) * 100).round(2)

Missing Email Handling

There are 20 missing email addresses (4% of the cleaned dataset). These values were retained as missing rather than imputed because a customer's email cannot be reliably inferred from the available data. The associated customer records were retained because their other information remains useful for analysis.

In [ ]:
df[df['region'].isna()] # Inspect entries with missing values for region

Region: 60 records (12%) have missing region information. The missing values were retained rather than imputed because region cannot be reliably inferred from the available fields. The affected customer records were retained so that their other information remains available for analysis. Missing region data will be treated as an identified data-quality limitation


Signup dates: The signup_date field was converted to datetime using mixed-format parsing. Two invalid values (31/31/2025 and not_a_date) could not be converted and were set to NaT. These records were retained rather than deleted because their other customer information remains useful. They will be excluded from time-based signup analysis because their signup dates cannot be reliably determined. Future signup dates were flagged for review rather than automatically removed.

In [ ]:
df[df['age'].isna()] #Inspect entries with missing values for 'region'

In [ ]:
df.groupby('gender')['age'].mean()

In [ ]:
df.groupby('region')['age'].mean()

### Age handling

Ages outside the plausible 18–100 validation range are treated as missing because their correct values cannot be established from the available data. Missing ages are retained rather than imputed to avoid introducing unsupported values.

### Email cleaning and validation

In [ ]:
email_pattern = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'

invalid_emails = df[
    df['email'].notna() &
    ~df['email'].str.match(email_pattern, na=False)
]

invalid_emails[['customer_id', 'email']]

In [ ]:
len(invalid_emails)

In [ ]:
# Clean the email adresses
df['email'] = df['email'].str.strip()
df['email'] = df['email'].str.replace('_at_', '@', regex=False)

In [ ]:
# confirm that the email addresses have been cleaned
invalid_emails = df[ df['email'].notna() & ~df['email'].str.match(email_pattern, na=False)]

len(invalid_emails)

## 4. Cleaning audit and business impact

In [ ]:
#Create a data frame of count and percenatage of missing values
missing_summary = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing Percentage': (df.isna().mean() * 100).round(2)
})

missing_summary

In [ ]:
duplicate_summary = pd.DataFrame({
    "Measure": [
        "Records before deduplication",
        "Duplicate rows removed",
        "Records after deduplication",
        "Duplicate customer IDs remaining"
    ],
    "Count": [
        records_before_dedup,
        duplicate_rows_removed,
        len(df),
        int(df["customer_id"].duplicated().sum())
    ]
})
display(duplicate_summary)

In [ ]:
category_corrections = pd.DataFrame({
    'Column': [
        'plan_selected',
        'plan_selected',
        'plan_selected',
        'gender'
    ],
    'Original Value': [
        'Pro ',
        'basic ',
        'Premuim',
        'male / female / MALE / FEMALE'
    ],
    'Corrected Value': [
        'Pro',
        'Basic',
        'Premium',
        'Male / Female'
    ],
    'Cleaning Rule': [
        'Remove whitespace',
        'Remove whitespace and standardise capitalisation',
        'Correct spelling error',
        'Remove whitespace and standardise capitalisation'
    ]
})

category_corrections

### Category and Email Corrections

| Field | Issue identified | Records affected | Action taken |
|---|---|---:|---|
| `plan_selected` | Inconsistent capitalisation, spacing and one spelling error (`Premuim`) | Multiple | Removed whitespace, standardised capitalisation, and corrected `Premuim` → `Premium` |
| `gender` | Inconsistent capitalisation and spacing | Multiple | Removed whitespace and standardised capitalisation |
| `email` | Malformed addresses using `_at_` instead of `@` and/or surrounding whitespace | 15 | Removed surrounding whitespace and replaced `_at_` with `@` where the intended address was unambiguous |

**Validation:** After cleaning the email addresses, the email validation check returned `0` invalid addresses.

## Business Impact and Severity

| Data quality issue | Severity | Business impact |
|---|---|---|
| Missing `region` (60 records; 12%) | **High** | Limits the reliability of regional customer analysis and may affect regional marketing and onboarding decisions. |
| Missing `email` (20 records; 4%) | **Medium** | Prevents direct email communication with affected customers and may limit onboarding or marketing activities. |
| Malformed `email` addresses (15 records) | **Medium** | Could result in failed customer communications if the addresses were used without correction. |
| Missing `age` (20 records; 4%) | **Medium** | Reduces the completeness of age-based customer analysis and may affect age-group comparisons. |
| Duplicate customer records (16 rows removed) | **Medium** | Could inflate customer counts and distort analysis by source, region, plan or other customer characteristics. |
| Future signup dates | **Medium** | May distort time-based signup analysis if future dates are treated as genuine historical signups. |
| Inconsistent `plan_selected` values | **Low** | Could split the same plan into separate categories and produce inaccurate plan-level counts. |
| Inconsistent `gender` values | **Low** | Could split customers into duplicate categories and affect gender-based marketing analysis. |

### Severity Prioritisation

Severity was prioritised according to the potential impact of each issue on the project's required analyses and business decisions. Missing `region` data was classified as **High** because it affects 12% of records and directly affects regional analysis. Missing and malformed email data were classified as **Medium** because they can affect customer communication. Missing age and duplicate records were also classified as **Medium** because they can affect customer segmentation and analytical accuracy. Formatting inconsistencies were classified as **Low** because they were corrected during cleaning and do not remain in the final dataset.

## 5. Exploratory analysis

In [ ]:
# Exclude invalid and future dates from historical time-series analysis.
valid_historical_signups = df.loc[
    df["signup_date"].notna() &
    (df["signup_date"] <= analysis_date)
].copy()

weekly_signups = (
    valid_historical_signups
    .set_index("signup_date")
    .resample("W-SUN")
    .size()
    .rename("signup_count")
)
weekly_signups.index.name = "week_ending"

print(f"Customers included in weekly analysis: {weekly_signups.sum():,}")
print(
    f"Historical date range: {valid_historical_signups['signup_date'].min().date()} "
    f"to {valid_historical_signups['signup_date'].max().date()}"
)
display(weekly_signups.to_frame().tail(20))

In [ ]:
df['signup_date'].min(), df['signup_date'].max()

In [ ]:
weekly_signups.sum()

**Interpretation**

Weekly signups are calculated using valid, non-future signup dates. Records with invalid dates are retained in the cleaned dataset but excluded from the time-series aggregation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly_signups.index, weekly_signups.values)
ax.set_title("Weekly Customer Signups")
ax.set_xlabel("Week Ending")
ax.set_ylabel("Number of Signups")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "weekly_signups.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

### Interpretation

Weekly signup activity was very low between 2019 and 2023, with most weeks recording no signups. Signup activity increased substantially from 2024 onwards, with more frequent signups throughout 2025 and 2026. The highest observed weekly total was 10 signups.

The analysis includes 498 customers with valid signup dates. Two customers were excluded from the weekly aggregation because their signup dates were invalid and were converted to `NaT` during data cleaning. Future-dated signup records were retained but flagged for data-quality review, so the apparent decline toward the end of the period should be interpreted cautiously.

In [ ]:
#Signup by source

source_summary = pd.DataFrame({
    'Signups': df['source'].value_counts(),
    'Percentage': (df['source'].value_counts(normalize=True) * 100).round(2)
})
source_summary

In [ ]:
source_chart = source_summary.sort_values("Signups")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(source_chart.index.astype(str), source_chart["Signups"])
ax.set_title("Customer Signups by Acquisition Source")
ax.set_xlabel("Number of Signups")
ax.set_ylabel("Acquisition Source")

for i, value in enumerate(source_chart["Signups"]):
    ax.text(value + 2, i, str(value), va="center")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "acquisition_sources.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
#Signup by region
region_summary = pd.DataFrame({
    'Signups': df['region'].value_counts(dropna=False),
    'Percentage': (
        df['region']
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(2)
    )
})

region_summary

**Interpretation**

North America has the largest number of recorded customers, with 146 customers (29.2% of the dataset), followed by Europe with 124 customers (24.8%). Asia-Pacific accounts for 90 customers (18.0%), while Latin America and Middle East & Africa account for 46 (9.2%) and 34 (6.8%) customers respectively.

Region data is missing for 60 customers (12.0%). These records were retained as missing because their region could not be reliably inferred from the available data. Consequently, regional comparisons should be interpreted with some caution.

In [ ]:
#Signup by plan
plan_summary = pd.DataFrame({
    'Signups': df['plan_selected'].value_counts(),
    'Percentage': (
        df['plan_selected']
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
})

plan_summary

In [ ]:
# Market opt_in breakdowm
marketing_by_gender = pd.crosstab(
    df['gender'],
    df['marketing_opt_in']
)

marketing_by_gender

In [ ]:
marketing_by_gender_pct = pd.crosstab(
    df['gender'],
    df['marketing_opt_in'],
    normalize='index'
).mul(100).round(2)

marketing_by_gender_pct

In [ ]:
import matplotlib.pyplot as plt

opt_in_rates = marketing_by_gender_pct['Yes'].sort_values()

plt.figure(figsize=(8, 5))

plt.bar(
    opt_in_rates.index,
    opt_in_rates.values
)

plt.title('Marketing Opt-in Rate by Gender')
plt.xlabel('Gender')
plt.ylabel('Opt-in Rate (%)')

for i, value in enumerate(opt_in_rates.values):
    plt.text(i, value + 1, f'{value:.1f}%', ha='center')

plt.ylim(0, 70)
plt.tight_layout()
plt.show()

**Interpretation**

Marketing opt-in rates varied across the gender groups. The Other group had the highest observed opt-in rate at 60.87%, followed by Male customers at 50.65% and Female customers at 47.15%.

However, the Other group contains only 23 customers, compared with 246 Female and 231 Male customers. Therefore, the higher opt-in rate for the Other group should be interpreted cautiously because it is based on a much smaller sample.

Overall, the results suggest some differences in observed marketing opt-in behaviour by gender, although further analysis would be needed to determine whether these differences are meaningful.

In [ ]:
age_summary = pd.Series({
    'Minimum Age': df['age'].min(),
    'Maximum Age': df['age'].max(),
    'Mean Age': round(df['age'].mean(), 2),
    'Median Age': round(df['age'].median(), 2),
    'Missing Ages': df['age'].isna().sum()
})

age_summary

In [ ]:
# Inspecting 'Age' data

invalid_ages = df[
    (df['age'] < 18) |
    (df['age'] > 100)
]

invalid_ages[['customer_id', 'age']].sort_values('age')

In [ ]:
# Treat ages outside the plausible range of 18–100 as missing
df.loc[
    (df['age'] < 18) | (df['age'] > 100),
    'age'
] = pd.NA

In [ ]:
df[df['customer_id'].isin(
    ['CU1054', 'CU1349', 'CU1151', 'CU1486', 'CU1024']
)][['customer_id', 'age']]

In [ ]:
# 'Age' stats recalculated after data cleaning

age_summary = pd.Series({
    'Minimum Age': df['age'].min(),
    'Maximum Age': df['age'].max(),
    'Mean Age': round(df['age'].mean(), 2),
    'Median Age': df['age'].median(),
    'Missing Ages': df['age'].isna().sum()
})

age_summary

### Interpretation

After cleaning, customer ages range from 18 to 65 years. The mean age is 33.32 years, while the median age is 32 years, indicating that the typical customer is in their early thirties.

There are 25 missing age values, representing 5.0% of the 500 customer records. This includes the original missing values and five implausible age values that were converted to missing because their correct ages could not be reliably determined.

For the analysis, ages below 18 or above 100 were treated as implausible. Rather than estimating replacement values, these records were retained with missing ages to avoid introducing unsupported assumptions into age-based analysis.

## 6. Business question analysis

# 4. Business Question Analysis

This section uses the cleaned customer dataset to answer the four business questions defined in the project brief.

The analysis focuses on:
1. The acquisition source with the most users in the latest month with recorded signup activity.
2. The region showing the highest proportion of incomplete customer records.
3. Whether marketing opt-in rates vary across age groups.
4. The most common subscription plan overall and how plan preferences vary by age group.

In [ ]:
# Analysis to Answer Question 1
today = pd.Timestamp.today().normalize()

latest_valid_date = df.loc[
    df['signup_date'] <= today,
    'signup_date'
].max()

latest_valid_date

In [ ]:
# Replaced by the dynamic latest-month analysis above.

In [ ]:
monthly_signups = (
    df.loc[df['signup_date'] <= today, 'signup_date']
      .dt.to_period('M')
      .value_counts()
      .sort_index()
)

monthly_signups

In [ ]:
# Replaced by the dynamic latest-month analysis above.

### Interpretation

The result is calculated dynamically using the latest month containing valid, non-future signup activity. If that month contains very few signups, the result should be treated as descriptive rather than representative of normal acquisition performance.

In [ ]:
#Analysis to answer question 2
#Counting the total number of missing data for each region
missing_by_region = df.groupby('region').agg(
    customers=('customer_id', 'size'),
    missing_email=('email', lambda x: x.isna().sum()),
    missing_signup_date=('signup_date', lambda x: x.isna().sum()),
    missing_age=('age', lambda x: x.isna().sum())
)

missing_by_region


In [ ]:
df['incomplete_record'] = (
    df[['email', 'signup_date', 'age']]
    .isna()
    .any(axis=1)
)

region_completeness = (
    df[df['region'].notna()]
    .groupby('region')
    .agg(
        customers=('customer_id', 'size'),
        affected_records=('incomplete_record', 'sum')
    )
)

region_completeness['affected_pct'] = (
    region_completeness['affected_records']
    / region_completeness['customers']
    * 100
).round(2)

region_completeness.sort_values('affected_pct', ascending=False)
q2_winner = region_completeness.index[0]
q2_rate = float(region_completeness.iloc[0]["affected_pct"])

In [ ]:
#Answer to Question 3
# put customers's age into age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 24, 34, 44, 54, 65],
    labels=['18–24', '25–34', '35–44', '45–54', '55–65']
)

In [ ]:
#determine the opt-in rate for each age group
age_opt_in = pd.crosstab(
    df['age_group'],
    df['marketing_opt_in'],
    normalize='index'
).mul(100).round(2)

age_opt_in

In [ ]:
# Answer to question 4
plan_by_age_group = pd.crosstab(
    df['age_group'],
    df['plan_selected']
)

plan_by_age_group
q4_overall_plan = plan_summary["Signups"].idxmax()
q4_overall_count = int(plan_summary.loc[q4_overall_plan, "Signups"])

In [ ]:
import matplotlib.pyplot as plt

opt_in_rates = age_opt_in['Yes']

plt.figure(figsize=(9, 5))

plt.bar(opt_in_rates.index, opt_in_rates.values)

plt.title('Marketing Opt-in Rate by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Marketing Opt-in Rate (%)')

for i, value in enumerate(opt_in_rates.values):
    plt.text(
        i,
        value + 1,
        f'{value:.1f}%',
        ha='center'
    )

plt.ylim(0, 80)
plt.tight_layout()
plt.show()

**Interpretation — marketing opt-in by age group**

The chart shows the percentage of customers in each age group who opted into marketing.

The 18–24 age group had the highest marketing opt-in rate at 68.4%, followed by 25–34 at 52.9%. The rate generally decreased across the older age groups, reaching a low of 30.0% among customers aged 45–54, before increasing slightly to 38.9% among customers aged 55–65.

Overall, the results suggest that younger customers were generally more likely to opt into marketing than older customers. However, the relationship is not perfectly linear, and the results show an association rather than evidence that age directly causes differences in marketing preferences.

## 7. Key findings

The summary below is generated from the cleaned dataset so that the portfolio notebook remains aligned with the actual analysis.

In [ ]:
print("KEY FINDINGS")
print("=" * 60)

print(
    f"1. Latest valid activity month: {latest_month}. "
    f"Top acquisition source: {q1_winner} ({q1_count} signup(s))."
)
print(
    f"2. Region with the highest proportion of incomplete records: "
    f"{q2_winner} ({q2_rate:.2f}%)."
)
print(
    f"3. Age-group marketing opt-in ranges from "
    f"{age_opt_in['Yes'].min():.1f}% to {age_opt_in['Yes'].max():.1f}%."
)
print(
    f"4. Most common plan overall: {q4_overall_plan} "
    f"({q4_overall_count} customers; "
    f"{plan_summary.loc[q4_overall_plan, 'Percentage']:.1f}%)."
)
print(
    f"5. Final customer records: {len(df):,}; "
    f"duplicate IDs remaining: {df['customer_id'].duplicated().sum()}."
)

## 8. Business recommendations

1. **Improve data-entry validation.** Add validation rules at source to reduce missing region, email and age values and prevent invalid dates.
2. **Strengthen duplicate prevention.** Use customer identifiers and system-level controls to reduce repeated customer records before they reach downstream reporting.
3. **Improve email validation.** Validate email structure at data capture and prevent ambiguous or malformed addresses from being stored.
4. **Monitor acquisition over complete periods.** The latest-month source result should not be treated as a performance benchmark when the month contains very few signups. Use multiple complete periods and, where available, conversion, acquisition cost and customer-value measures.
5. **Investigate age-related marketing patterns.** The observed differences in opt-in rates can inform targeted testing, but campaigns should be evaluated using sufficient sample sizes and controlled comparisons.
6. **Review plan preferences by segment.** Differences in plan selection across age groups may support targeted messaging or product positioning, subject to validation with larger datasets.

## 9. Limitations

- Missing values limit some demographic and regional comparisons.
- Invalid signup dates are excluded from time-based analysis.
- Future-dated records are retained for data-quality review but excluded from historical trend analysis.
- The latest-month acquisition result may be based on very few signups and therefore has limited representativeness.
- Age-group comparisons are descriptive and do not establish causation.
- The dataset does not include acquisition cost, conversion, revenue or customer lifetime value, so channel performance cannot be evaluated on business value alone.

## 10. Final validation

The checks below confirm that the cleaned dataset meets the main assumptions used by the analysis.

### Reproducibility

For a minimal Python environment, install:

```text
pandas
matplotlib
```

The notebook is designed to run in Google Colab or a standard Python/Jupyter environment.

In [ ]:
assert len(df) == df["customer_id"].nunique(), "Duplicate customer IDs remain."
assert len(invalid_emails) == 0, "Invalid non-missing emails remain."
assert df["signup_date"].dtype.kind == "M", "signup_date is not datetime."
assert set(df["plan_selected"].dropna().unique()).issubset({"Basic", "Pro", "Premium"}),     "Unexpected plan category remains."

print("All final validation checks passed.")
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Portfolio charts saved to: {OUTPUT_DIR.resolve()}")